# 01. Quality Control with scLucid

This notebook is a practical, end-to-end tour of the scLucid QC module using real datasets in `data/`.

It demonstrates:

- Standalone QC metric calculation
- Data-driven threshold recommendation
- Standard workflow execution
- Hierarchical multi-sample threshold policy
- User threshold overrides
- Tumor-aware QC behavior
- QC trace, review summary, decision table, evidence chain, and shared evidence bundle
- Profile-aware benchmark assessment with retention, marker-fidelity, risk level, reasons, and action items
- Sample/cell-type retention bias checks for multi-sample exploratory analysis
- JSON / Markdown / HTML report outputs

The default run uses `data/pbmc3k.h5ad`. A smaller optional PDAC tumor example is included later.

## 0. Runtime Setup

The notebook configures cache directories before importing Scanpy/scLucid. This prevents common `numba`, matplotlib, and joblib cache issues in fresh environments or CI-like sessions.

In [ ]:
from pathlib import Path
import os
import tempfile

# Resolve project root whether the notebook is run from repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "01_quality_control"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = Path(tempfile.gettempdir()) / "sclucid_notebook_cache"
for name in ["numba", "mpl", "joblib", "xdg"]:
    (CACHE_ROOT / name).mkdir(parents=True, exist_ok=True)

os.environ.setdefault("NUMBA_CACHE_DIR", str(CACHE_ROOT / "numba"))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "mpl"))
os.environ.setdefault("JOBLIB_TEMP_FOLDER", str(CACHE_ROOT / "joblib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_ROOT / "xdg"))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("SCLUCID_SAFE_PARALLEL", "1")

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Output dir:   {OUT_DIR}")

## 1. Imports

Import scLucid before heavy Scanpy work so the package runtime setup is applied.

In [ ]:
import json
from pprint import pprint

import numpy as np
import pandas as pd
from IPython.display import display

import scLucid
import scanpy as sc
from scLucid.config import set_config
from scLucid.qc import (
    AdaptiveThresholdLearner,
    DoubletConfig,
    FilterConfig,
    MarkingConfig,
    MetricsReportingConfig,
    QCThresholds,
    QCWorkflowConfig,
    calculate_qc_metric,
    compute_marker_fidelity,
    compute_retention_metrics,
    evaluate_qc_benchmark,
    filter_cells,
    generate_qc_html_report,
    mark_low_quality_cell,
    recommend_intelligent_qc,
    render_qc_benchmark_compact_markdown,
    run_standard_qc,
)

set_config(n_jobs=2, verbosity=1)
sc.settings.verbosity = 2

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

## 2. Load Real PBMC Data

The repository includes `data/pbmc3k.h5ad`. This is small enough for a notebook but still useful for demonstrating multi-sample-aware QC because it has `sampleID` annotations.

In [ ]:
pbmc_path = DATA_DIR / "pbmc3k.h5ad"
adata = sc.read_h5ad(pbmc_path)

if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

print(adata)
print("obs columns:", list(adata.obs.columns))
print("layers:", list(adata.layers.keys()))
print("samples:", adata.obs["sampleID"].value_counts().to_dict() if "sampleID" in adata.obs else "missing")

## 3. Standalone QC Metric Calculation

Use `calculate_qc_metric` when you want metrics without running the full workflow. In real exploratory analysis this is often the first quick inspection step.

In [ ]:
adata_metrics = adata.copy()
adata_metrics = calculate_qc_metric(
    adata_metrics,
    sample_key="sampleID",
    show_plots=False,
    plot_top_genes=False,
    plot_violin=False,
    plot_scatter=False,
    export_stats=False,
    print_stats=False,
    calculate_cell_cycle=False,
)

qc_cols = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]
adata_metrics.obs[qc_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

## 4. Adaptive Threshold Learning

The lower-level adaptive threshold learner is useful for inspecting one metric at a time. The full workflow uses richer recommendation logic, but this gives a transparent view of threshold behavior.

In [ ]:
learner = AdaptiveThresholdLearner(method="percentile")
thresholds = {
    "min_genes_percentile": learner.learn_threshold(
        adata_metrics.obs["n_genes_by_counts"].to_numpy(),
        metric_name="n_genes_by_counts",
        direction="lower",
    ),
    "max_mt_percentile": learner.learn_threshold(
        adata_metrics.obs["pct_counts_mt"].to_numpy(),
        metric_name="pct_counts_mt",
        direction="upper",
    ),
}
thresholds

## 5. Intelligent QC Recommendation

`recommend_intelligent_qc` converts data distributions into executable recommendations with confidence intervals and evidence. This is the part that replaces fixed one-size-fits-all cutoffs.

In [ ]:
recommendation = recommend_intelligent_qc(
    adata_metrics.copy(),
    tissue_type="pbmc",
    strategy="auto",
    plot=False,
)

rec = recommendation.to_dict()
pd.DataFrame(
    [
        {
            "parameter": name,
            "threshold": payload.get("threshold"),
            "ci_lower": payload.get("ci_lower"),
            "ci_upper": payload.get("ci_upper"),
            "confidence": payload.get("confidence"),
            "method": payload.get("method"),
        }
        for name, payload in rec.items()
        if isinstance(payload, dict) and "threshold" in payload
    ]
)

In [ ]:
print("Strategy:", rec["overall_strategy"])
print("Overall confidence:", rec["overall_confidence"])
print("Data quality score:", rec["data_quality_score"])
print("Concerns:")
pprint(rec["concerns"])

## 6. Full Standard QC Workflow

This is the recommended end-user path. The config below is intentionally non-interactive and notebook-safe:

- `use_recommendations=True`: apply intelligent recommendations unless the user explicitly overrides a threshold.
- `threshold_mode="hierarchical"`: compute per-sample adaptive thresholds when multiple samples are present.
- doublet algorithm and heuristic detection are disabled for speed in this tutorial; the config block shows where to enable them.

In [ ]:
pbmc_out = OUT_DIR / "pbmc3k_standard"

qc_config = QCWorkflowConfig(
    sample_key="sampleID",
    species="human",
    tissue="blood",
    tissue_type="pbmc",
    save_dir=str(pbmc_out),
    use_recommendations=True,
    threshold_mode="hierarchical",
    use_parallel=False,
    metrics_reporting_config=MetricsReportingConfig(
        show_plots=False,
        plot_top_genes=False,
        plot_violin=False,
        plot_scatter=False,
        export_stats=True,
        print_stats=True,
    ),
    marking_config=MarkingConfig(
        show_plots=False,
        plot_outliers=False,
    ),
    doublet_config=DoubletConfig(
        # For production, set run_algorithm=True and choose scrublet/solo/doubletdetection.
        run_algorithm=False,
        use_heuristics=False,
        show_plots=False,
        plot_summary=False,
        plot_bar=False,
        plot_scatter=False,
        plot_upset=False,
        export_stats=False,
    ),
    filter_config=FilterConfig(
        criteria_to_filter=["outlier_min_genes", "outlier_mt", "predicted_doublet"],
        combination_logic="any",
    ),
)

adata_qc = run_standard_qc(adata, config=qc_config, show_progress=False)

print(f"Original cells: {adata.n_obs}")
print(f"Filtered cells: {adata_qc.n_obs}")
print(f"Retention: {adata_qc.n_obs / adata.n_obs:.1%}")
print(f"Output: {pbmc_out}")

## 7. Inspect the Unified QC Trace

scLucid stores all QC provenance under `adata.uns["sclucid"]["qc"]`. The most useful user-facing object is `review_summary["data"]`.

In [ ]:
qc_trace = adata_qc.uns["sclucid"]["qc"]
print("Stored QC trace keys:")
print(sorted(qc_trace.keys()))

review = qc_trace["review_summary"]["data"]
print("Review summary keys:")
print(sorted(review.keys()))

### Decision Table

The decision table is the compact answer to: “what threshold was used, where did it come from, and what evidence supports it?”

In [ ]:
decision_table = pd.DataFrame(review["decision_table"])
decision_table[
    [
        "parameter",
        "recommended",
        "applied",
        "source",
        "recommendation_method",
        "confidence",
        "ci_lower",
        "ci_upper",
        "is_filtering_enabled",
    ]
]

### Evidence Chain, Evidence Bundle, and Output Health

These fields are designed for review and reproducibility. They make it easy to see if a run is safe to pass downstream.

`evidence_chain` is QC-specific and ordered by workflow stage. `evidence_bundle` is the shared scLucid evidence schema that later modules can also emit.

In [ ]:
pprint(review["execution_trace"])
print("\nOutput health:")
pprint(review["output_health"])
print("\nEvidence chain:")
pprint(review["evidence_chain"])

evidence_bundle = review["evidence_bundle"]
print("\nShared evidence bundle:")
pprint({
    "schema_version": evidence_bundle["schema_version"],
    "module": evidence_bundle["module"],
    "stage": evidence_bundle["stage"],
    "status": evidence_bundle["status"],
    "confidence": evidence_bundle["confidence"],
    "n_decisions": len(evidence_bundle["decisions"]),
    "n_evidence_items": len(evidence_bundle["evidence_chain"]),
    "n_action_items": len(evidence_bundle["action_items"]),
    "related_review_keys": evidence_bundle["related_review_keys"],
})

## 8. Benchmark Assessment: Retention, Marker Fidelity, and Risk

The QC benchmark layer evaluates the output without changing filtering decisions. It supports PBMC, tissue, tumor, and cell-line profiles.

The important object is now `benchmark_summary["assessment"]`: it converts raw checks into a publication-facing status, risk level, reasons, and review actions.

In [ ]:
benchmark = review["benchmark_summary"]
assessment = benchmark["assessment"]

print("Profile:", benchmark["profile_label"])
print("Status:", benchmark["status"])
print("Risk level:", assessment["risk_level"])
print("Retention:", benchmark["retention"]["retention_rate"])
print("Marker fidelity available:", benchmark["marker_fidelity"]["available"])
print("Assessment:", assessment["summary"])

display(pd.DataFrame(benchmark["checks"])[
    ["name", "passed", "severity", "value", "threshold", "interpretation", "recommendation"]
])
pd.DataFrame(assessment["recommendations"])

In [ ]:
# Standalone benchmark API: useful for comparing two QC strategies or external QC outputs.
standalone_benchmark = evaluate_qc_benchmark(
    adata_before=adata,
    adata_after=adata_qc,
    tissue_type="pbmc",
    sample_key="sampleID",
)
print(render_qc_benchmark_compact_markdown(standalone_benchmark))
standalone_benchmark["retention"]

## 9. Generated Reports and Artifacts

When `save_dir` is set, the workflow writes JSON/Markdown review bundles and an HTML report. These files are intended to make the analysis auditable outside the notebook.

In [ ]:
for path in sorted(pbmc_out.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".json", ".md", ".html", ".csv", ".xlsx"}:
        print(path.relative_to(PROJECT_ROOT))

In [ ]:
# Optional: generate a standalone HTML report explicitly.
html_path = OUT_DIR / "pbmc3k_qc_report_explicit.html"
generate_qc_html_report(
    adata_qc,
    adata_before=adata,
    output_path=str(html_path),
    title="PBMC3K QC Report",
)
html_path

## 10. Manual Threshold Override Path

Professional users often need to override one threshold while accepting recommendations for others. scLucid records that explicitly in `user_override_summary` and the `decision_table` source column.

In [ ]:
override_out = OUT_DIR / "pbmc3k_user_override"

override_config = qc_config.model_copy(deep=True)
override_config.save_dir = str(override_out)
override_config.marking_config.thresholds = QCThresholds(
    min_genes=300,  # explicit user override
    pc_mt=20.0,     # explicit user override
)

override_qc = run_standard_qc(adata, config=override_config, show_progress=False)
override_review = override_qc.uns["sclucid"]["qc"]["review_summary"]["data"]

print("Retention:", override_qc.n_obs / adata.n_obs)
pprint(override_review["user_override_summary"])

pd.DataFrame(override_review["decision_table"])[
    ["parameter", "recommended", "applied", "source", "is_filtering_enabled"]
]

## 11. Low-Level Mark and Filter API

The workflow is usually preferable, but the individual functions are available for custom exploratory workflows.

In [ ]:
manual = adata.copy()
manual = calculate_qc_metric(
    manual,
    sample_key="sampleID",
    show_plots=False,
    plot_top_genes=False,
    plot_violin=False,
    plot_scatter=False,
    export_stats=False,
    print_stats=False,
    calculate_cell_cycle=False,
)
manual = mark_low_quality_cell(
    manual,
    sample_key="sampleID",
    config=MarkingConfig(
        show_plots=False,
        plot_outliers=False,
        thresholds=QCThresholds(min_genes=200, pc_mt=20.0),
    ),
)
manual_filtered = filter_cells(
    manual,
    config=FilterConfig(criteria_to_filter=["outlier_min_genes", "outlier_mt"], combination_logic="any"),
    copy=True,
)

print(manual_filtered)
print(manual_filtered.uns.get("sclucid", {}).get("qc", {}).get("filtering_results", {}))

## 12. Optional Tumor-Aware QC on Real PDAC Data

The repository also includes real PDAC datasets. This example uses a downsampled PDAC object to demonstrate tumor-aware behavior without making the notebook too slow.

Tumor-aware mode does not mean “run tumor module”; it means QC should treat tumor tissue properties carefully, especially elevated mitochondrial signal and heterogeneous populations.

In [ ]:
RUN_TUMOR_EXAMPLE = True
pdac_path = DATA_DIR / "schlesinger2020.pdac.h5ad"

if RUN_TUMOR_EXAMPLE and pdac_path.exists():
    pdac = sc.read_h5ad(pdac_path)
    if pdac.n_obs > 1500:
        rng = np.random.default_rng(61)
        idx = rng.choice(pdac.n_obs, size=1500, replace=False)
        pdac = pdac[idx].copy()
    if "counts" not in pdac.layers:
        pdac.layers["counts"] = pdac.X.copy()
    print(pdac)
    print(pdac.obs[["sampleID", "group", "study"]].head())
else:
    pdac = None
    print("Tumor example skipped.")

In [ ]:
if pdac is not None:
    tumor_out = OUT_DIR / "pdac_tumor_aware"
    tumor_config = QCWorkflowConfig(
        sample_key="sampleID",
        species="human",
        tissue="pancreas",
        tissue_type="pancreatic_tumor",
        save_dir=str(tumor_out),
        use_recommendations=True,
        threshold_mode="hierarchical",
        use_parallel=False,
        metrics_reporting_config=MetricsReportingConfig(
            show_plots=False,
            plot_top_genes=False,
            plot_violin=False,
            plot_scatter=False,
            export_stats=False,
            print_stats=False,
        ),
        marking_config=MarkingConfig(show_plots=False, plot_outliers=False),
        doublet_config=DoubletConfig(
            run_algorithm=False,
            use_heuristics=False,
            show_plots=False,
            export_stats=False,
        ),
        filter_config=FilterConfig(
            criteria_to_filter=["outlier_min_genes", "predicted_doublet"],
            combination_logic="any",
        ),
    )
    pdac_qc = run_standard_qc(pdac, config=tumor_config, show_progress=False)
    tumor_review = pdac_qc.uns["sclucid"]["qc"]["review_summary"]["data"]
    print(f"PDAC retention: {pdac_qc.n_obs / pdac.n_obs:.1%}")
    pprint(tumor_review["tumor_aware_summary"])
    pprint(tumor_review["benchmark_summary"]["criteria"])
    print("Tumor benchmark assessment:")
    pprint(tumor_review["benchmark_summary"]["assessment"])
    pd.DataFrame(tumor_review["decision_table"])[
        ["parameter", "recommended", "applied", "source", "is_filtering_enabled"]
    ]
else:
    print("Tumor workflow skipped.")

## 13. What to Inspect Before Moving to Preprocessing

Use this checklist before passing `adata_qc` to preprocessing:

1. `output_health["status"]` is `ok`, or any issues are understood.
2. `benchmark_summary["assessment"]["risk_level"]` is acceptable for the biological context.
3. Benchmark `reasons` and `recommendations` have been reviewed when status is not `pass`.
4. Retention is not implausibly low/high and sample/cell-type retention bias checks are understood.
5. User overrides, if any, are intentional and recorded in the decision table.
6. Tumor datasets have tumor-aware notes reviewed before hard-filtering high-MT populations.
7. The exported `qc_review_summary.json`, `qc_benchmark.json`, and `evidence_bundle` are kept with analysis results.

In [ ]:
summary = {
    "pbmc_cells_before": adata.n_obs,
    "pbmc_cells_after": adata_qc.n_obs,
    "pbmc_retention": adata_qc.n_obs / adata.n_obs,
    "pbmc_review_status": review["output_health"]["status"],
    "pbmc_benchmark_status": review["benchmark_summary"]["status"],
    "pbmc_benchmark_risk": review["benchmark_summary"]["assessment"]["risk_level"],
    "pbmc_qc_readiness": review["qc_readiness"]["status"],
    "output_dir": str(OUT_DIR),
}
summary